In [1]:
import pysam
import pandas as pd
import re

In [2]:
wk_dir = "/working/lab_olgak/lijunX/OV_meBRCA1_Nanopore"
icgc_ov_sample_sheet = "/working/genomeinfo/share/analysis/AOCS_-_ICGC_ovarian_cancer_project/AOCS_-_ICGC_ovarian_cancer_project.StudyPairedAnalysis_20220407.tsv"
icgc_ov_dir = pd.read_csv (icgc_ov_sample_sheet, sep = '\t')

scott_sample_sheet = "/working/genomeinfo/share/analysis/Scott_-_Ovarian_cancer.wdl/Scott_-_Ovarian_cancer.StudySomaticDnaAnalysis_20230313.tsv"
scott_pdx_dir = pd.read_csv(scott_sample_sheet, sep = "\t")



In [5]:
scott_pdx_df = []
for index, row in scott_pdx_dir.iterrows():
    entry = []
    #tumour
    entry.append(row["testSampleSubmittedWithID"])
    entry.append(row["testSampleType"])
    bam_read = pysam.AlignmentFile(row["testAlignedReadGroupSet"], "rb").fetch("chr17", 41280066,41280069)
    cigar = []
    softclip_cigar = []
    softclip_count = 0
    for reads in bam_read:
        cigar.append(reads.cigarstring)
        if "S" in reads.cigarstring:
            softclip_cigar.append(re.search(r'(\d+)S', reads.cigarstring).group(1))
            softclip_count += 1
    entry.append(','.join(cigar))
    entry.append(str(len(cigar)))
    entry.append(str(softclip_count))
    entry.append(','.join(softclip_cigar))
    #control
    entry.append(row["controlSampleLabel"])
    entry.append(row["controlSampleType"])
    bam_read = pysam.AlignmentFile(row["controlAlignedReadGroupSet"], "rb").fetch("chr17", 41280066,41280069)
    cigar = []
    softclip_cigar = []
    softclip_count = 0
    for reads in bam_read:
        cigar.append(reads.cigarstring)
        if "S" in reads.cigarstring:
            softclip_cigar.append(re.search(r'(\d+)S', reads.cigarstring).group(1))
            softclip_count += 1
    entry.append(','.join(cigar))
    entry.append(str(len(cigar)))
    entry.append(str(softclip_count))
    entry.append(','.join(softclip_cigar))
    
    scott_pdx_df.append(entry)
    

[W::hts_idx_load3] The index file is older than the data file: /mnt/lustre/working/genomeinfo/sample/c/2/c2cf5c96-7f1f-497c-abfd-a0b5d308e0a8/aligned_read_group_set/8e15230f-6a88-48a6-af88-c297a5e0d6a3.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /mnt/lustre/working/genomeinfo/sample/8/0/8014bb5b-becf-45b3-ae19-557509787aad/aligned_read_group_set/37424c46-7f79-4f6f-8b37-e51f902c793d.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /mnt/lustre/working/genomeinfo/sample/c/2/c2cf5c96-7f1f-497c-abfd-a0b5d308e0a8/aligned_read_group_set/a23f651e-9494-4788-b4bb-b40285e70521.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /mnt/lustre/working/genomeinfo/sample/3/8/38a4df72-6ff8-491f-b374-5d346f1fbae3/aligned_read_group_set/d5279847-c97c-4ba0-88b2-573507bd4bff.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /mnt/lustre/working/genomeinfo/sample/0/a/0aba19f9-49f8-4871-9320-111e11b294a0/aligned_read_group

In [12]:
41280068, 41280533

print(scott_pdx_dir["testAlignedReadGroupSet"][1])
bam_read = pysam.AlignmentFile(scott_pdx_dir["testAlignedReadGroupSet"][1], "rb").fetch("chr17", 41280068,41280533)

/mnt/lustre/working/genomeinfo/sample/8/0/8014bb5b-becf-45b3-ae19-557509787aad/aligned_read_group_set/37424c46-7f79-4f6f-8b37-e51f902c793d.bam


[W::hts_idx_load3] The index file is older than the data file: /mnt/lustre/working/genomeinfo/sample/8/0/8014bb5b-becf-45b3-ae19-557509787aad/aligned_read_group_set/37424c46-7f79-4f6f-8b37-e51f902c793d.bam.bai


In [6]:
df = pd.DataFrame(scott_pdx_df)
print(df)

                    0                                    1   \
0            134169-BL                       Primary tumour   
1       134169-7182-T3  Mouse xenograft derived from tumour   
2       134148-6414-T3  Mouse xenograft derived from tumour   
3            134148-BL                       Primary tumour   
4              PH039-T                       Primary tumour   
5       PH039-13288-T5  Mouse xenograft derived from tumour   
6       PH039-13368-T5  Mouse xenograft derived from tumour   
7            134027-BL                       Primary tumour   
8           134027-p34     Cell line derived from xenograft   
9       134027-3165-T3  Mouse xenograft derived from tumour   
10     134027H-7302-T4  Mouse xenograft derived from tumour   
11         134056-2272  Mouse xenograft derived from tumour   
12        SFRC01217-S1                       Primary tumour   
13         SFRC01122-T                       Primary tumour   
14      SFRC01122S1-T3  Mouse xenograft derived from tu

In [72]:
# load vcf files to extract deletion
vcf_file = wk_dir + "/output/"+SampleID+"/deepvariants/"+SampleID+".vcf.gz"
deepvariants = pysam.VariantFile(vcf_file)
deepvariantsFile = deepvariants.fetch("chr17", 43128051-50,43128516+50)


bamfile_path = wk_dir+"/output/"+SampleID+"/"+SampleID+"_filtered_mapped.bam"


In [73]:
%%capture cap --no-stderr
match = list()
discordant = list()
for variants in deepvariantsFile:
    ref_allele, alt_allele = variants.alleles
    if len(alt_allele) >= 100:
        print("chrom:", variants.chrom, " start:", variants.start, " stop:", variants.stop)
        print("alt:",alt_allele)
        bamfile =  pysam.AlignmentFile(bamfile_path, "rb").fetch("chr17", variants.start-200,variants.stop+200)
        for reads in bamfile:
            read_pos = reads.get_reference_positions(full_length =True)
            pos_1 = read_pos.index(variants.start-1)
            pos_2 = read_pos.index(variants.start+2)
            if read_pos[pos_1:pos_2].count(None)>=100:
                discordant.append(reads.query_name)
            else:
                match.append(reads.query_name)

print("match:", ",".join(match))
print("discordant:",",".join(discordant))

In [74]:
with open(wk_dir+"/output/"+SampleID+"LTR_readID.txt", 'w') as f:
    f.write(cap.stdout)